# Notebook chạy lại thí nghiệm trên nhiều môi trường

Notebook tự tìm hoặc sao chép repository, tải dữ liệu Git LFS nếu có, kiểm tra và cài thư viện phụ thuộc phù hợp với môi trường, cố định seed, kiểm tra dữ liệu rồi chạy lần lượt các notebook thí nghiệm. Mọi đường dẫn đều được tạo tương đối từ thư mục làm việc hoặc thư mục gốc của repository.

Notebook này luôn dùng hồ sơ và embedding LLM đã lưu trong repository; tùy chọn tạo lại hồ sơ được cố định tắt (`REGENERATE_LLM_PROFILES = False`). Vì vậy notebook không tải lại mô hình Qwen 7B để sinh hồ sơ. Huấn luyện các mô hình thí nghiệm vẫn có thể mất nhiều thời gian; chạy trên CPU thường chậm hơn đáng kể.

Để tái lập đúng phiên bản mã nguồn, hãy đặt `REPRO_REPO_REF` thành mã commit SHA của repository trước khi sao chép. Mặc định notebook dùng nhánh `draft` như bản trước và ghi SHA thực tế vào tệp kê khai của lần chạy.

In [ ]:
import importlib.util
import os
import platform
import sys
from pathlib import Path

REPO_URL = os.environ.get(
    "REPRO_REPO_URL",
    "https://github.com/datachain-uit/CN-23520430-HanLTN-DiemLK.git",
)
REPO_REF = os.environ.get("REPRO_REPO_REF", "draft")
REPO_NAME = "CN-23520430-HanLTN-DiemLK"
SEED = int(os.environ.get("REPRO_SEED", "2026"))

INSTALL_DEPENDENCIES = True
# Kaggle và Colab đã cung cấp sẵn phần lớn môi trường Python. Mặc định giữ
# các phiên bản thư viện có sẵn; chỉ áp dụng phiên bản ghim nghiêm ngặt khi được yêu cầu.
REGENERATE_LLM_PROFILES = False  # Dùng hồ sơ và embedding đã lưu; không tạo lại hồ sơ LLM.
RUN_EXPERIMENTS = os.environ.get("RUN_EXPERIMENTS", "1").strip().lower() not in {"0", "false", "no"}


def detect_runtime():
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or os.environ.get("KAGGLE_URL_BASE"):
        return "Kaggle"
    try:
        import google.colab  # noqa: F401  # Chỉ dùng để nhận diện môi trường Colab
        return "Colab"
    except Exception:
        pass
    if os.environ.get("VSCODE_PID") or os.environ.get("TERM_PROGRAM") == "vscode":
        return "Máy cá nhân / VS Code"
    return f"Máy cá nhân / Jupyter ({platform.system()})"


RUNTIME = detect_runtime()
_default_strict = "0" if RUNTIME in {"Kaggle", "Colab"} else "1"
STRICT_PINS = os.environ.get("REPRO_STRICT_PINS", _default_strict).strip().lower() in {"1", "true", "yes"}
print(f"Môi trường: {RUNTIME} | Python: {platform.python_version()} | Hệ điều hành: {platform.platform()}")
print(f"Tạo lại hồ sơ LLM: {REGENERATE_LLM_PROFILES} | Chạy lại toàn bộ thí nghiệm: {RUN_EXPERIMENTS}")

In [ ]:
import os
import subprocess
from pathlib import Path


def find_repository(start: Path):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "README.md").is_file() and (candidate / "data").is_dir() and (candidate / "src").is_dir():
            return candidate
    return None


def run_command(args, *, cwd=None, env=None):
    args = [str(arg) for arg in args]
    print("\n$", subprocess.list2cmdline(args) if os.name == "nt" else " ".join(args))
    return subprocess.run(args, cwd=cwd, env=env, check=True, text=True)


configured_root = os.environ.get("REPRO_REPO_PATH")
if configured_root:
    REPO_ROOT = Path(configured_root).expanduser().resolve()
    if not ((REPO_ROOT / "README.md").is_file() and (REPO_ROOT / "data").is_dir()):
        raise FileNotFoundError(f"REPRO_REPO_PATH không trỏ tới thư mục gốc repository hợp lệ: {REPO_ROOT}")
else:
    REPO_ROOT = find_repository(Path.cwd())
    if REPO_ROOT is None:
        clone_dir = Path.cwd().resolve() / "workspace" / REPO_NAME
        clone_dir.parent.mkdir(parents=True, exist_ok=True)
        if clone_dir.exists():
            raise FileExistsError(
                f"Thư mục đích sao chép đã tồn tại nhưng không phải repository hợp lệ: {clone_dir}. "
                "Hãy đặt REPRO_REPO_PATH trỏ tới một bản sao repository hợp lệ hoặc chọn thư mục làm việc khác."
            )
        if len(REPO_REF) == 40 and all(ch in "0123456789abcdefABCDEF" for ch in REPO_REF):
            run_command(["git", "clone", REPO_URL, clone_dir])
            run_command(["git", "fetch", "--depth", "1", "origin", REPO_REF], cwd=clone_dir)
            run_command(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=clone_dir)
        else:
            run_command(["git", "clone", "--branch", REPO_REF, "--single-branch", REPO_URL, clone_dir])
        REPO_ROOT = clone_dir.resolve()

os.environ["DATA_DIR"] = str(REPO_ROOT)
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Thư mục gốc repository: {REPO_ROOT}")
if (REPO_ROOT / ".git").exists():
    try:
        git_lfs = subprocess.run(["git", "lfs", "version"], cwd=REPO_ROOT, capture_output=True, text=True)
        if git_lfs.returncode != 0:
            raise RuntimeError(
                "Các tệp dữ liệu nghiên cứu dung lượng lớn cần Git LFS. Hãy cài Git LFS, khởi động lại kernel rồi chạy lại notebook."
            )
        print(git_lfs.stdout.strip())
        run_command(["git", "lfs", "install"], cwd=REPO_ROOT)
        run_command(["git", "lfs", "pull"], cwd=REPO_ROOT)
    except FileNotFoundError as exc:
        raise RuntimeError("Không tìm thấy Git. Hãy cài Git và Git LFS rồi chạy lại notebook.") from exc
else:
    print("Không tìm thấy thư mục .git; giả định đây là bản xuất repository đã có sẵn dữ liệu đầy đủ.")

if (REPO_ROOT / ".git").exists():
    REPO_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_ROOT, text=True).strip()
else:
    REPO_COMMIT = "không có thông tin (bản xuất repository)"
print(f"Phiên bản repository được chọn: {REPO_COMMIT}")

In [ ]:
import importlib.metadata as metadata
import json
import os
import random
import re
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

LOCK_FILE = REPO_ROOT / "requirements.txt"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DIR = REPO_ROOT / "results" / "reproduction_runs" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)


def installed_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None


def locked_packages(path):
    packages = []
    for raw in path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or line.startswith("-"):
            continue
        match = re.match(r"^([A-Za-z0-9_.-]+)==([^;\s]+)", line)
        if match:
            packages.append((match.group(1), match.group(2)))
    return packages


def pip_install(specs, extra_args=()):
    if not specs:
        return
    command = [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--upgrade-strategy", "only-if-needed"]
    command.extend(extra_args)
    command.extend(specs)
    run_command(command, cwd=REPO_ROOT)


package_status = {}
if INSTALL_DEPENDENCIES:
    if not LOCK_FILE.is_file():
        raise FileNotFoundError(f"Không tìm thấy tệp khóa phiên bản thư viện phụ thuộc: {LOCK_FILE}")
    missing_or_mismatched = []
    host_mismatches = []
    for name, version in locked_packages(LOCK_FILE):
        if name.lower() == "bitsandbytes" and not REGENERATE_LLM_PROFILES:
            package_status[name] = installed_version(name) or "chưa cài (thư viện tùy chọn để tạo hồ sơ LLM)"
            continue
        if name.lower() == "torch":
            continue  # Cài đặt hoặc kiểm tra PyTorch riêng để chọn bản phù hợp với nền tảng.
        current = installed_version(name)
        package_status[name] = current
        if current == version:
            continue
        if current is not None and RUNTIME in {"Kaggle", "Colab"} and not STRICT_PINS:
            host_mismatches.append((name, current, version))
        else:
            missing_or_mismatched.append(f"{name}=={version}")

    if missing_or_mismatched:
        pip_install(missing_or_mismatched)
    if host_mismatches:
        print("\nĐang dùng các phiên bản thư viện do nền tảng cung cấp để tránh làm hỏng môi trường notebook được quản lý:")
        for name, current, expected in host_mismatches:
            print(f"  {name}: phiên bản đã cài {current}; phiên bản thí nghiệm yêu cầu {expected}")
        print("Để áp dụng chính xác các phiên bản đã ghim, hãy dùng môi trường cục bộ mới hoặc đặt REPRO_STRICT_PINS=1.")

    torch_pin = next((v for n, v in locked_packages(LOCK_FILE) if n.lower() == "torch"), "2.11.0")
    torch_base = torch_pin.split("+", 1)[0]
    torch_current = installed_version("torch")
    package_status["torch"] = torch_current
    has_nvidia = shutil.which("nvidia-smi") is not None
    current_cuda_build = bool(torch_current and "+cu128" in torch_current)
    should_replace_torch = torch_current is None or (
        STRICT_PINS
        and (torch_current.split("+", 1)[0] != torch_base or (has_nvidia and not current_cuda_build))
    )
    if should_replace_torch:
        if has_nvidia:
            pip_install([f"torch=={torch_pin}"], ["--extra-index-url", "https://download.pytorch.org/whl/cu128"])
        elif platform.system() in {"Windows", "Linux"}:
            pip_install([f"torch=={torch_base}"], ["--index-url", "https://download.pytorch.org/whl/cpu"])
        else:
            pip_install([f"torch=={torch_base}"])
    if REGENERATE_LLM_PROFILES:
        bnb_current = installed_version("bitsandbytes")
        bnb_pin = next((v for n, v in locked_packages(LOCK_FILE) if n.lower() == "bitsandbytes"), None)
        if bnb_pin and bnb_current != bnb_pin:
            pip_install([f"bitsandbytes=={bnb_pin}"])
    # nbconvert không nằm trong danh sách thư viện của thí nghiệm nhưng cần để chạy các notebook thành phần.
    nbconvert_version = installed_version("nbconvert")
    if nbconvert_version is None or tuple(int(x) for x in nbconvert_version.split(".")[:2]) < (7, 16):
        pip_install(["nbconvert>=7.16,<8"])
else:
    print("Đã tắt cài đặt thư viện phụ thuộc; notebook sẽ tiếp tục dùng môi trường kernel hiện tại.")

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
random.seed(SEED)
try:
    import numpy as np
    np.random.seed(SEED)
except ImportError:
    np = None
try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except (AttributeError, TypeError):
        pass
except ImportError:
    torch = None
try:
    import tensorflow as tf
    tf.random.set_seed(SEED)
except Exception as exc:
    tf = None
    print(f"Bỏ qua việc cố định seed cho TensorFlow ({type(exc).__name__}: {exc})")

# Ghi lại phiên bản thực tế (cùng phiên bản đã ghim) để nhận biết khác biệt giữa các môi trường.
for package, _ in locked_packages(LOCK_FILE):
    package_status[package] = installed_version(package)
for package in ["numpy", "pandas", "pyarrow", "scikit-learn", "scipy", "sentence-transformers", "transformers", "accelerate", "bitsandbytes", "torch", "nbconvert"]:
    package_status[package] = installed_version(package)
manifest = {
    "run_id": RUN_ID,
    "runtime": RUNTIME,
    "os": platform.platform(),
    "python": platform.python_version(),
    "seed": SEED,
    "repository": REPO_URL,
    "requested_ref": REPO_REF,
    "resolved_commit": REPO_COMMIT,
    "strict_pins": STRICT_PINS,
    "packages": package_status,
    "torch_cuda_available": bool(torch is not None and torch.cuda.is_available()),
    "torch_cuda_version": None if torch is None else torch.version.cuda,
}
(RUN_DIR / "environment_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"Đã cố định seed ở {SEED}; tệp kê khai môi trường được lưu tại {RUN_DIR / 'environment_manifest.json'}")
if torch is not None:
    print(f"PyTorch: {torch.__version__} | Có CUDA: {torch.cuda.is_available()} | Số thiết bị: {torch.cuda.device_count()}")

In [ ]:
from pathlib import Path

required_files = [
    Path("data/full_interactions/interactions_full.parquet"),
    Path("data/raw_data_processed/user-course.csv"),
    Path("data/course_llm/course_data.parquet"),
    Path("data/splits/cold_scenario/train_cold.parquet"),
    Path("data/splits/cold_scenario/val_cold.parquet"),
    Path("data/splits/cold_scenario/test_cold.parquet"),
]
if not REGENERATE_LLM_PROFILES:
    required_files.extend([
        Path("data/course_llm/1_embeddings/raw_course_text_embeddings.npy"),
        Path("data/course_llm/1_embeddings/llm_profile_embeddings.npy"),
        Path("data/course_llm/1_embeddings/llm_profile_no_prereq_followup_embeddings.npy"),
    ])
missing_files = [str(path) for path in required_files if not (REPO_ROOT / path).is_file()]
lfs_pointers = []
for relative in required_files:
    path = REPO_ROOT / relative
    if path.is_file():
        try:
            if path.open("rb").read(120).startswith(b"version https://git-lfs.github.com/spec/v1"):
                lfs_pointers.append(str(relative))
        except OSError:
            pass
if missing_files or lfs_pointers:
    details = []
    if missing_files:
        details.append("Thiếu các tệp dữ liệu bắt buộc: " + ", ".join(missing_files))
    if lfs_pointers:
        details.append("Chưa tải nội dung các tệp Git LFS sau: " + ", ".join(lfs_pointers))
    raise FileNotFoundError("\n".join(details) + "\nHãy bật Git LFS và kết nối mạng, sau đó chạy lại ô thiết lập repository.")
print(f"Kiểm tra dữ liệu ban đầu thành công: tìm thấy đủ {len(required_files)} tệp bắt buộc.")

In [ ]:
import shutil
from pathlib import Path


def run_python(*args):
    command = [sys.executable, *map(str, args)]
    env = os.environ.copy()
    env["DATA_DIR"] = str(REPO_ROOT)
    env["PYTHONHASHSEED"] = str(SEED)
    env["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    run_command(command, cwd=REPO_ROOT, env=env)


if REGENERATE_LLM_PROFILES:
    if torch is None or not torch.cuda.is_available():
        raise RuntimeError(
            "Tạo lại hồ sơ Qwen 7B theo quy trình 4-bit đã ghim phiên bản cần GPU NVIDIA hỗ trợ CUDA. "
            "Hãy dùng hồ sơ đã lưu (mặc định) hoặc bật GPU tương thích trước khi đặt REGENERATE_LLM_PROFILES=True."
        )
    profile_dir = Path("data/course_llm/2_profiles_full_and_shards")
    for shard in (0, 1):
        run_python(
            "llm/course_profile.py",
            "--data_path", "data/course_llm/course_data.parquet",
            "--prompt_path", "src/prompts/prompt_template.txt",
            "--output_dir", profile_dir,
            "--num_shards", "2",
            "--shard", str(shard),
        )
    run_python(
        "llm/course_profile.py",
        "--data_path", "data/course_llm/course_data.parquet",
        "--output_dir", profile_dir,
        "--num_shards", "2",
        "--merge_shards",
    )
    embedding_dir = REPO_ROOT / "data/course_llm/1_embeddings"
    embedding_dir.mkdir(parents=True, exist_ok=True)
    for artifact in list((REPO_ROOT / profile_dir).glob("*embeddings*.npy")) + list((REPO_ROOT / profile_dir).glob("*course_index.json")):
        shutil.copy2(artifact, embedding_dir / artifact.name)
    run_python(
        "llm/course_profile.py",
        "--data_path", "data/course_llm/course_data.parquet",
        "--prompt_path", "src/prompts/prompt_template.txt",
        "--output_dir", "data/course_llm/1_embeddings",
        "--encode_raw_only",
    )
    run_python(
        "llm/course_profile.py",
        "--data_path", "data/course_llm/course_data.parquet",
        "--prompt_path", "src/prompts/prompt_template.txt",
        "--output_dir", "data/course_llm/3_quality_evaluation",
        "--masked_prereq_eval",
    )
else:
    print("Đang dùng các hồ sơ và embedding có sẵn trong repository hoặc được cung cấp qua LFS; bỏ qua bước tạo hồ sơ.")

In [ ]:
import hashlib
import subprocess
from pathlib import Path


def run_script(relative_path):
    path = REPO_ROOT / relative_path
    if not path.is_file():
        raise FileNotFoundError(f"Không tìm thấy script trong quy trình: {path}")
    run_python(path)


def run_notebook(relative_path):
    source = REPO_ROOT / relative_path
    if not source.is_file():
        raise FileNotFoundError(f"Không tìm thấy notebook trong quy trình: {source}")
    command = [
        sys.executable, "-m", "nbconvert",
        "--to", "notebook", "--execute",
        "--ExecutePreprocessor.timeout=-1",
        "--output", f"{source.stem}-executed.ipynb",
        "--output-dir", str(RUN_DIR),
        str(source),
    ]
    env = os.environ.copy()
    env["DATA_DIR"] = str(REPO_ROOT)
    env["PYTHONHASHSEED"] = str(SEED)
    env["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    run_command(command, cwd=REPO_ROOT, env=env)


if RUN_EXPERIMENTS:
    print("Đang kiểm tra tính toàn vẹn của repository và các phép kiểm tra ở phụ lục...")
    run_script("src/eval/check_cold_items.py")
    run_script("src/eval/random_ndcg_analytic.py")
    run_script("src/eval/time_consistent_subset.py")

    backbone_files = [
        REPO_ROOT / "data/embeddings/01_lightgcn_backbone_cold/g_u.npy",
        REPO_ROOT / "data/embeddings/01_lightgcn_backbone_cold/g_i.npy",
    ]
    for path in backbone_files:
        if path.is_file():
            digest = hashlib.md5(path.read_bytes()).hexdigest()
            print(f"MD5 {path.relative_to(REPO_ROOT)}: {digest}")

    notebooks = [
        "src/models/lightgcn_leakage.ipynb",
        "src/models/cold_scenario/lightgcn_backbone_cold.ipynb",
        "src/eval/evaluate_test_random.ipynb",
        "src/models/cold_scenario/lightgcn_id_cold.ipynb",
        "src/models/cold_scenario/content-knn-v2.ipynb",
        "src/models/cold_scenario/dropoutnet-v2.ipynb",
        "src/models/cold_scenario/dropoutnet-v2-p-rq3.ipynb",
        "src/models/cold_scenario/dropoutnet-llm-3d.ipynb",
        "src/models/coldrec-adapter.ipynb",
        "src/models/semproj_experiments.ipynb",
        "src/models/coldrec-adapter-usim-raw.ipynb",
        "results/final-stats.ipynb",
    ]
    for item in notebooks:
        print(f"\n=== Đang chạy {item} ===")
        run_notebook(item)
    print(f"\nĐã hoàn tất các bước được yêu cầu. Notebook sau khi chạy được lưu tại: {RUN_DIR}")
else:
    print("Đang tắt chạy lại thí nghiệm. Đặt RUN_EXPERIMENTS=1 để chạy toàn bộ quy trình.")